In [1]:
# !pip install qiskit qiskit-aer matplotlib

In [2]:
# 导入 Qiskit 相关库
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# 1. 创建一个包含 2 个量子比特和 2 个经典比特（用于测量）的量子线路
qc = QuantumCircuit(2, 2)

# 2. 对第 0 号比特施加 Hadamard 门（H 门）
# H 门的作用是让比特进入“0”和“1”的叠加态（Superposition），此时有 50% 的概率是 0，50% 的概率是 1
qc.h(0)

# 3. 对 0 号和 1 号比特施加受控非门（CNOT 门），以 0 号为控制位，1 号为目标位
# 这一步是魔法的开始：它让两个比特产生“量子纠缠”。
# 如果 0 号是 0，1 号保持不变（也是0）；如果 0 号是 1，1 号翻转（变成1）。
qc.cx(0, 1)

# 4. 测量两个量子比特，并将结果保存到经典比特中
qc.measure([0, 1], [0, 1])

# 5. 绘制出你的量子线路图
print("--- 量子线路图 ---")
print(qc.draw(output='text'))

# 6. 使用本地的经典模拟器（AerSimulator）来运行这个线路
simulator = AerSimulator()
# 运行 1024 次实验（shots）
job = simulator.run(qc, shots=1024)
result = job.result()

# 7. 获取测量结果并绘制直方图
counts = result.get_counts(qc)
print("\n--- 实验测量结果（1024次运行） ---")
print(counts)

# 绘制结果
plot_histogram(counts)
plt.show()

--- 量子线路图 ---
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 

--- 实验测量结果（1024次运行） ---
{'00': 519, '11': 505}


In [3]:
from qiskit import QuantumCircuit, transpile
from qiskit.providers.fake_provider import GenericBackendV2

# 1. 创建最新的虚拟芯片（它模拟了 IBM 真实 Heron/Eagle 芯片的物理参数）
backend = GenericBackendV2(num_qubits=2)

# 2. 编写你的理想量子线路
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

# 3. 编译（Transpile）：将理想门翻译为硬件真正支持的“原生门（Native Gates）”
# 真实超导芯片在物理上其实只支持极少数几种物理脉冲（比如 rz 虚拟门，sx 根号X脉冲，以及 cz/ecr 纠缠脉冲）
transpiled_qc = transpile(qc, backend)

print("=== 1. 理想门在硬件底层的【原生门翻译】 ===")
print("（你会发现 H 门和 CX 门被替换为了 sx, rz, cz 等原生门）")
print(transpiled_qc.draw(output='text'))

# 4. 从新版 backend.target 中直接抓取底层的【物理参数】
target = backend.target

# 查询 0 号比特上单比特原生门（sx 门，物理上是 90 度的微波脉冲）的持续时间和误差
sx_props = target['sx'][(0,)]
print("\n=== 2. 单比特物理脉冲参数 (sx 门) ===")
print(f"物理脉冲持续时间: {sx_props.duration * 1e9:.2f} 纳秒 (ns)")
print(f"物理脉冲控制误差: {sx_props.error * 100:.4f}%")

# 查询双比特原生门（如 cz 门或 ecr 门，物理上是让两个比特共振的脉冲）
# 动态检测芯片支持哪种双比特物理脉冲
native_2q_gate = 'cz' if 'cz' in target else 'ecr' if 'ecr' in target else 'cx'
two_q_props = target[native_2q_gate][(0, 1)]
print(f"\n=== 3. 双比特纠缠物理脉冲参数 ({native_2q_gate} 门) ===")
print(f"物理脉冲持续时间: {two_q_props.duration * 1e9:.2f} 纳秒 (ns)")
print(f"物理脉冲控制误差: {two_q_props.error * 100:.4f}%")

=== 1. 理想门在硬件底层的【原生门翻译】 ===
（你会发现 H 门和 CX 门被替换为了 sx, rz, cz 等原生门）
global phase: π/4
         ┌─────────┐┌────┐┌─────────┐     
q_0 -> 0 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π/2) ├──■──
         └─────────┘└────┘└─────────┘┌─┴─┐
q_1 -> 1 ────────────────────────────┤ X ├
                                     └───┘

=== 2. 单比特物理脉冲参数 (sx 门) ===
物理脉冲持续时间: 59.05 纳秒 (ns)
物理脉冲控制误差: 0.0091%

=== 3. 双比特纠缠物理脉冲参数 (cx 门) ===
物理脉冲持续时间: 748.58 纳秒 (ns)
物理脉冲控制误差: 0.1698%


In [4]:
!which python

/opt/homebrew/anaconda3/bin/python
